In [ ]:
from nrem_sc.constants import PROCESSED_DATA_PATH, INTERIM_DATA_PATH
from nrem_sc.utils import hd_tuning

import numpy as np
import pynapple as nap
import seaborn as sns

import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.colors import ListedColormap

from cmap import Colormap

# Load data
unit_id = "116b"
sleep_states    = nap.load_file(PROCESSED_DATA_PATH / unit_id / "sleep.npz")
hd_spikes       = nap.load_file(PROCESSED_DATA_PATH / unit_id / "hd_spikes_filtered.npz")
hd_angle        = np.deg2rad(nap.load_file(PROCESSED_DATA_PATH / unit_id / "angle_openfield.npz"))
sessions        = nap.load_file(PROCESSED_DATA_PATH / unit_id / "sessions_labeled.npz")

Colormap for cyclic data

In [ ]:
cm = Colormap('cmocean:phase')
mpl.colormaps.register(cm.to_mpl())
cm

Compute the tuning curves (Bayesian decoding)

In [ ]:
from scipy.ndimage import gaussian_filter1d

tcs = nap.compute_tuning_curves(
    data=hd_spikes.restrict(hd_angle.time_support), features=hd_angle, bins=60,
    epochs=hd_angle.time_support, range=(0.0, 2*np.pi), feature_names=['head_direction']
    )
tcs.values = gaussian_filter1d(tcs.values, sigma=3, axis=1, mode="wrap")
pref_ang = tcs.idxmax(dim="head_direction")

In [ ]:

spikes = hd_spikes.restrict(ep)
idx = spikes[spikes['rate'] > 0].index
tcs, pref_ang = hd_tuning(hd_spikes, hd_angle)
norm_tcs = (tcs - tcs.min(dim='head_direction')) / (tcs.max(dim='head_direction') - tcs.min(dim='head_direction'))
norm_tcs = norm_tcs.sortby(pref_ang)